# dbApps06a Task: Primary Keys, Foreign Keys & Redundancy

**Course:** Database Applications Development  
**Medina County Career Center**

---

**4 tasks — start easy, finish strong.**

**Need help?**
- Review the **Walkthrough** (`dbApps06a_Walkthrough.ipynb`) — we did similar queries together
- Check the **Data Dictionary** (`imdb_data_dictionary.md`) for friendly table names and columns
- Check the **Schema Chart** (`imdb_schema_chart.md`) for the full table/column/PK/FK map
- Use the **Study Guide** (`dbApps06_StudyGuide.md`) for vocabulary and syntax

**Key Concepts:**
- **Primary Key (PK)** = unique ID for each row (no duplicates allowed)
- **Foreign Key (FK)** = a column that points to another table's PK
- **Redundancy** = storing the same data over and over (bad!)
- `COUNT(DISTINCT col)` = counts only unique values

In [1]:
# Setup - Run this cell first
import pandas as pd
import sqlite3

dbPath = 'imdb_class_2010plus.db'
connection = sqlite3.connect(dbPath)
cursor = connection.cursor()

print('Connected to IMDb database!')

Connected to IMDb database!


---

## Task 1: Spot the Redundancy (Easy)

Run the query below to see all credits for Inception from the **Cast & Crew Credits** table (`title_principals`). Notice how the same **Title ID** (`tconst`) repeats for every single credit row.

In a bad design where the full movie title, year, and genre were stored in *every* row, all of that info would be duplicated over and over.

**After running it, answer:** How many rows have the same Title ID? Why would it be a problem if every row also stored the title, year, and genre?

> *Walkthrough reference:* We looked at this exact table in Part 3 of the walkthrough.

In [2]:
# Task 1: Look at all credits for Inception (just run this!)
query = '''
    SELECT tconst, nconst, category, characters
    FROM title_principals
    WHERE tconst = 'tt1375666'
    ORDER BY ordering
'''

df = pd.read_sql_query(query, connection)
print(f'Credits for Inception ({len(df)} rows):')
print(df.to_string(index=False))

Credits for Inception (19 rows):
   tconst    nconst            category              characters
tt1375666 nm0000138               actor                ["Cobb"]
tt1375666 nm0330687               actor              ["Arthur"]
tt1375666 nm0680983               actor             ["Ariadne"]
tt1375666 nm0913822               actor               ["Saito"]
tt1375666 nm0362766               actor               ["Eames"]
tt1375666 nm2438307               actor               ["Yusuf"]
tt1375666 nm0614165               actor ["Robert Fischer, Jr."]
tt1375666 nm0000297               actor            ["Browning"]
tt1375666 nm0182839             actress                 ["Mal"]
tt1375666 nm0000592               actor     ["Maurice Fischer"]
tt1375666 nm0634240            director                    None
tt1375666 nm0634240              writer                    None
tt1375666 nm0634240            producer                    None
tt1375666 nm0858799            producer                    None
tt13756

**Your Answer:** How many rows have the same Title ID? Why is it a problem if every row also stored the title, year, and genre?



In [ ]:
all of them, because it wouldn't be sorted properly

---

## Task 2: Measure the Full Database Redundancy (Medium)

Now let's measure this problem across the *entire* database, not just one movie.

**Write a query** on the `title_principals` table that returns two numbers:
1. `COUNT(*)` as `totalCredits` — total number of credit rows
2. `COUNT(DISTINCT tconst)` as `uniqueTitles` — how many unique movies

The difference between those two numbers = how many duplicate copies of title info a bad one-table design would create.

> *Walkthrough reference:* We ran this exact query in Part 3 of the walkthrough — the big redundancy calculation.

In [4]:
# Task 2: Count total redundancy across the whole database
# Hint: SELECT COUNT(*) as totalCredits, COUNT(DISTINCT tconst) as uniqueTitles
#       FROM title_principals
query = '''
SELECT 
 COUNT(*) AS totalCredits, 
 COUNT(DISTINCT tconst) AS uniqueTitles
FROM 
    title_principals
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

 totalCredits  uniqueTitles
       364848         19451


**Your Answer:** Total credits? Unique titles? How many duplicate copies would a bad design create?
  364848         19451 and ther would be 34397 copies


---

## Task 3: Follow the Foreign Keys (Medium-Hard)

The raw `title_principals` data just shows IDs like `tt1375666` and `nm0000138`. Not very helpful! But those IDs are **foreign keys** — they point to other tables where the real info lives.

**Write a JOIN query** that follows the foreign keys to show real movie titles and actor names for a movie YOU pick.

Your query should:
- JOIN `title_principals p` → `title_basics b` (ON `p.tconst = b.tconst`) → `name_basics n` (ON `p.nconst = n.nconst`)
- Show: `b.primaryTitle`, `n.primaryName`, `p.category`
- Pick a movie: `WHERE b.primaryTitle = 'Your Movie'`
- Filter to main credits: `AND p.category IN ('actor', 'actress', 'director')`
- Sort by billing order: `ORDER BY p.ordering`

> *Walkthrough reference:* We did a 3-table JOIN just like this in Part 2 — first we looked at raw IDs, then JOINed to get real names. Use that as your template.

In [5]:
# Task 3: Use JOINs to follow the foreign keys and get real names
# JOIN chain: title_principals p → title_basics b (ON p.tconst = b.tconst)
#                                → name_basics n  (ON p.nconst = n.nconst)
# Pick a movie: WHERE b.primaryTitle = 'Your Movie'
# Filter: AND p.category IN ('actor', 'actress', 'director')
query = '''

'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

Empty DataFrame
Columns: [primaryTitle, primaryName, category]
Index: []


**Your Answer:** How does a successful JOIN prove that the foreign keys between these tables are valid?



---

## Task 4: The Update Problem (Challenge)

Pick 3-5 movies you like. Write a query that shows how many credit rows *each movie* has in `title_principals` (using a JOIN + GROUP BY).

This is the number of rows you'd have to update in a bad one-table design just to fix one typo in a movie title!

Your query should:
- JOIN `title_principals p` with `title_basics b` on `p.tconst = b.tconst`
- Pick your movies: `WHERE b.primaryTitle IN ('Movie1', 'Movie2', 'Movie3')`
- `GROUP BY b.primaryTitle`
- `COUNT(*) as rowsToUpdate`
- `ORDER BY rowsToUpdate DESC`

> *Walkthrough reference:* We did this for 5 popular movies in Part 3 of the walkthrough — the "update problem" query. Swap in YOUR movies.

In [6]:
# Task 4 (Challenge): How many rows would break if you had a bad design?
# Hint: JOIN title_principals p with title_basics b ON p.tconst = b.tconst
# Use: WHERE b.primaryTitle IN ('Movie1', 'Movie2', 'Movie3')
# GROUP BY b.primaryTitle and COUNT(*) as rowsToUpdate
query = '''
SELECT 
b.primaryTitle, 
COUNT(*) AS rowsToUpdate
FROM 
title_principals p
JOIN 
title_basics b ON p.tconst = b.tconst
WHERE 
b.primaryTitle IN ('The Matrix', 'Inception', 'Titanic', 'The Avengers', 'Interstellar')
GROUP BY 
 b.primaryTitle
ORDER BY 
rowsToUpdate DESC;
'''

df = pd.read_sql_query(query, connection)
print(df.to_string(index=False))

primaryTitle  rowsToUpdate
The Avengers            26
Interstellar            21
   Inception            19


**Your Answer:** How many rows would need updating for each movie? In our good design, how many rows would you update instead?

4

---

## You're Done!

You proved that:
- **Primary Keys** keep each row unique (no duplicate Title IDs)
- **Foreign Keys** connect tables safely (JOINs work because IDs match)
- **Redundancy** wastes space and creates risk (one typo = hundreds of broken rows!)

**Resources to keep handy:**
- **Data Dictionary** (`imdb_data_dictionary.md`)
- **Schema Chart** (`imdb_schema_chart.md`)
- **Study Guide** (`dbApps06_StudyGuide.md`)

In [7]:
# Clean up
connection.close()
print('Database connection closed.')

Database connection closed.
